<h1>02 - Tokens and Token Embeddings</h1>
<i>Exploring tokens and embeddings as an integral part of building LLMs</i>


In [1]:
# %%capture
# !pip install --upgrade transformers==4.41.2 sentence-transformers==3.0.1 gensim==4.3.2 scikit-learn==1.5.0 accelerate==0.31.0 peft==0.11.1 scipy==1.10.1 numpy==1.26.4

# Downloading and Running An LLM

The first step is to load our model onto the GPU for faster inference. Note that we load the model and tokenizer separately and keep them as such so that we can explore them separately.

In [2]:
import torch

In [3]:
from transformers import AutoModelForCausalLM, AutoTokenizer
model_path = "microsoft/Phi-3-mini-4k-instruct"
device = "mps" if torch.backends.mps.is_available() else "cpu"
# Load model and tokenizer
model = AutoModelForCausalLM.from_pretrained(
    model_path,
    device_map=device,
    torch_dtype="auto",
    trust_remote_code=False,
)
tokenizer = AutoTokenizer.from_pretrained("microsoft/Phi-3-mini-4k-instruct")

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

In [4]:
prompt = "Write an email apologizing to Sarah for the tragic gardening mishap. Explain how it happened.<|assistant|>"


In [5]:
input_ids = tokenizer(prompt, return_tensors="pt").input_ids.to('mps')
input_ids


tensor([[14350,   385,  4876, 27746,  5281,   304, 19235,   363,   278, 25305,
           293, 16423,   292,   286,   728,   481, 29889, 12027,  7420,   920,
           372,  9559, 29889, 32001]], device='mps:0')

In [6]:
device='mps'
# Tokenize the input prompt
encoded = tokenizer(prompt,padding=True, return_tensors="pt")

input_ids=encoded['input_ids'].to(device)
attention_mask=encoded['attention_mask'].to(device)
print(input_ids)
print(attention_mask)

# Generate the text
generation_output = model.generate(
  input_ids=input_ids,
  attention_mask=attention_mask,
  max_new_tokens=20
)

# # Print the output
# print(tokenizer.decode(generation_output[0]))

tensor([[14350,   385,  4876, 27746,  5281,   304, 19235,   363,   278, 25305,
           293, 16423,   292,   286,   728,   481, 29889, 12027,  7420,   920,
           372,  9559, 29889, 32001]], device='mps:0')
tensor([[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]],
       device='mps:0')


You are not running the flash-attention implementation, expect numerical differences.


In [7]:
model.generate??

Signature:
model.generate(
    inputs: Optional[torch.Tensor] = None,
    generation_config: Optional[transformers.generation.configuration_utils.GenerationConfig] = None,
    logits_processor: Optional[transformers.generation.logits_process.LogitsProcessorList] = None,
    stopping_criteria: Optional[transformers.generation.stopping_criteria.StoppingCriteriaList] = None,
    prefix_allowed_tokens_fn: Optional[Callable[[int, torch.Tensor], List[int]]] = None,
    synced_gpus: Optional[bool] = None,
    assistant_model: Optional[ForwardRef('PreTrainedModel')] = None,
    streamer: Optional[ForwardRef('BaseStreamer')] = None,
    negative_prompt_ids: Optional[torch.Tensor] = None,
    negative_prompt_attention_mask: Optional[torch.Tensor] = None,
    **kwargs,
) -> Union[transformers.generation.utils.GenerateDecoderOnlyOutput, transformers.generation.utils.GenerateEncoderDecoderOutput, transformers.generation.utils.GenerateBeamDecoderOnlyOutput, transformers.generation.utils.GenerateBeam

In [8]:
for id in input_ids[0]:
   print(f'id is:{id},\t,decoded_id:{tokenizer.decode(id)}')

id is:14350,	,decoded_id:Write
id is:385,	,decoded_id:an
id is:4876,	,decoded_id:email
id is:27746,	,decoded_id:apolog
id is:5281,	,decoded_id:izing
id is:304,	,decoded_id:to
id is:19235,	,decoded_id:Sarah
id is:363,	,decoded_id:for
id is:278,	,decoded_id:the
id is:25305,	,decoded_id:trag
id is:293,	,decoded_id:ic
id is:16423,	,decoded_id:garden
id is:292,	,decoded_id:ing
id is:286,	,decoded_id:m
id is:728,	,decoded_id:ish
id is:481,	,decoded_id:ap
id is:29889,	,decoded_id:.
id is:12027,	,decoded_id:Exp
id is:7420,	,decoded_id:lain
id is:920,	,decoded_id:how
id is:372,	,decoded_id:it
id is:9559,	,decoded_id:happened
id is:29889,	,decoded_id:.
id is:32001,	,decoded_id:<|assistant|>


In [9]:
generation_output

tensor([[14350,   385,  4876, 27746,  5281,   304, 19235,   363,   278, 25305,
           293, 16423,   292,   286,   728,   481, 29889, 12027,  7420,   920,
           372,  9559, 29889, 32001,  3323,   622, 29901,   317,  3742,   406,
          6225, 11763,   363,   278, 19906,   292,   341,   728,   481,    13,
            13,    13, 29928,   799]], device='mps:0')

In [10]:
print(tokenizer.decode(3323))
print(tokenizer.decode(622))
print(tokenizer.decode([3323, 622]))
print(tokenizer.decode(29901))

Sub
ject
Subject
:


# Comparing Trained LLM Tokenizers


In [11]:
from transformers import AutoModelForCausalLM, AutoTokenizer


In [12]:
background_colors=['41', '42', '43', '44', '45', '46']
def show_tokens(sentence, tokenizer_name):
    tokenizer=AutoTokenizer.from_pretrained(tokenizer_name)
    token_ids=tokenizer(sentence).input_ids
    for idx, t in enumerate(token_ids):
       standard_ansi=f'\x1b[{background_colors[idx%len(background_colors)]}m'
       reset_ansi='\x1b[0m'
       print(standard_ansi+tokenizer.decode(t)+reset_ansi,end='')

In [13]:
import numpy as np 

In [14]:
numbers=np.random.randint(0, 255, size=(6, 3))

In [15]:
for i in numbers:
    print(*i,end='')

234 36 21283 73 9542 34 21686 122 124125 138 90177 148 115

In [16]:
text = """
$ 😂
English and CAPITALIZATION
🎵 鸟
show_tokens False None elif == >= else: two tabs:"    " Three tabs: "       "
12.0*50=600
"""

In [17]:
show_tokens(text, "bert-base-uncased")

'(MaxRetryError('HTTPSConnectionPool(host=\'huggingface.co\', port=443): Max retries exceeded with url: /bert-base-uncased/resolve/main/tokenizer_config.json (Caused by NameResolutionError("HTTPSConnection(host=\'huggingface.co\', port=443): Failed to resolve \'huggingface.co\' ([Errno 8] nodename nor servname provided, or not known)"))'), '(Request ID: 38aa2215-21b6-46a0-be74-cad9d92935bf)')' thrown while requesting HEAD https://huggingface.co/bert-base-uncased/resolve/main/tokenizer_config.json
Retrying in 1s [Retry 1/5].
'(MaxRetryError('HTTPSConnectionPool(host=\'huggingface.co\', port=443): Max retries exceeded with url: /bert-base-uncased/resolve/main/tokenizer_config.json (Caused by NameResolutionError("HTTPSConnection(host=\'huggingface.co\', port=443): Failed to resolve \'huggingface.co\' ([Errno 8] nodename nor servname provided, or not known)"))'), '(Request ID: f6715f18-5b22-4576-9adc-8484164d5ed2)')' thrown while requesting HEAD https://huggingface.co/bert-base-uncased/res

[CLS]$[UNK]englishandcapital##ization[UNK][UNK]show_token##sfalsenoneeli##f==>=else:twotab##s:""threetab##s:""12.0*50=600[SEP]

In [18]:
show_tokens(text, "bert-base-cased")

'(MaxRetryError('HTTPSConnectionPool(host=\'huggingface.co\', port=443): Max retries exceeded with url: /bert-base-cased/resolve/main/tokenizer_config.json (Caused by NameResolutionError("HTTPSConnection(host=\'huggingface.co\', port=443): Failed to resolve \'huggingface.co\' ([Errno 8] nodename nor servname provided, or not known)"))'), '(Request ID: d171943e-8b9a-4a61-9629-7e584ef5f7e8)')' thrown while requesting HEAD https://huggingface.co/bert-base-cased/resolve/main/tokenizer_config.json
Retrying in 1s [Retry 1/5].
'(MaxRetryError('HTTPSConnectionPool(host=\'huggingface.co\', port=443): Max retries exceeded with url: /bert-base-cased/resolve/main/tokenizer_config.json (Caused by NameResolutionError("HTTPSConnection(host=\'huggingface.co\', port=443): Failed to resolve \'huggingface.co\' ([Errno 8] nodename nor servname provided, or not known)"))'), '(Request ID: 549adbe6-5316-4bb1-8555-48c1892f983a)')' thrown while requesting HEAD https://huggingface.co/bert-base-cased/resolve/mai

[CLS]$[UNK]EnglishandCA##PI##TA##L##I##Z##AT##ION[UNK][UNK]show_token##sF##als##eNoneel##if==>=else:twota##bs:""Threeta##bs:""12.0*50=600[SEP]

In [19]:
show_tokens(text, "gpt2")

'(MaxRetryError('HTTPSConnectionPool(host=\'huggingface.co\', port=443): Max retries exceeded with url: /gpt2/resolve/main/tokenizer_config.json (Caused by NameResolutionError("HTTPSConnection(host=\'huggingface.co\', port=443): Failed to resolve \'huggingface.co\' ([Errno 8] nodename nor servname provided, or not known)"))'), '(Request ID: c1000dfe-ee74-4797-93f3-ee0c79548fde)')' thrown while requesting HEAD https://huggingface.co/gpt2/resolve/main/tokenizer_config.json
Retrying in 1s [Retry 1/5].
'(MaxRetryError('HTTPSConnectionPool(host=\'huggingface.co\', port=443): Max retries exceeded with url: /gpt2/resolve/main/tokenizer_config.json (Caused by NameResolutionError("HTTPSConnection(host=\'huggingface.co\', port=443): Failed to resolve \'huggingface.co\' ([Errno 8] nodename nor servname provided, or not known)"))'), '(Request ID: f92fc95a-bf41-4c14-bd5b-ade173476cb2)')' thrown while requesting HEAD https://huggingface.co/gpt2/resolve/main/tokenizer_config.json
Retrying in 2s [Retr


$ ��
English and CAPITALIZATION
��� ���
show_tokens False None elif == >= else: two tabs:"    " Three tabs: "       "
12.0*50=600


In [20]:
## added by myself
tokenizer=AutoTokenizer.from_pretrained('gpt2')
tokenizer.decode([8582,236,113])

'🎵'

In [21]:
show_tokens(text, "google/flan-t5-small")

$<unk>EnglishandCAPITALIZATION<unk><unk>show_tokensFalseNoneelif==>=else:twotabs:""Threetabs:""12.0*50=600</s>

In [22]:
# The official is `tiktoken` but this the same tokenizer on the HF platform
show_tokens(text, "Xenova/gpt-4")

Special tokens have been added in the vocabulary, make sure the associated word embeddings are fine-tuned or trained.



$ ��
English and CAPITALIZATION
��� ���
show_tokens False None elif == >= else: two tabs:"    " Three tabs: "       "
12.0*50=600


In [23]:
# You need to request access before being able to use this tokenizer
show_tokens(text, "bigcode/starcoder2-15b")


$ ��
English and CAPITALIZATION
��� ��
show_tokens False None elif == >= else: two tabs:"    " Three tabs: "       "
12.0*50=600


In [24]:
show_tokens(text, "facebook/galactica-1.3b")


$ ����
English and CAPITALIZATION
���� ���
show_tokens False None elif == >= else: two tabs:"    " Three tabs: "       "
12.0*50=600


In [25]:
show_tokens(text, "microsoft/Phi-3-mini-4k-instruct")

Special tokens have been added in the vocabulary, make sure the associated word embeddings are fine-tuned or trained.



$����
EnglishandCAPITALIZATION
�������
show_tokensFalseNoneelif==>=else:twotabs:"  "Threetabs:"     "
12.0*50=600


# Contextualized Word Embeddings From a Language Model (Like BERT)

In [26]:
from transformers import AutoModel, AutoTokenizer

# Load a tokenizer
tokenizer = AutoTokenizer.from_pretrained("microsoft/deberta-base")

# Load a language model
model = AutoModel.from_pretrained("microsoft/deberta-v3-xsmall")

# Tokenize the sentence
tokens = tokenizer('Hello world', return_tensors='pt')

# Process the tokens
# Passing only input_ids using [0]
output = model(**tokens)[0]

In [27]:
output??

Type:        Tensor
String form:
tensor([[[-3.4816,  0.0861, -0.1819,  ..., -0.0612, -0.3911,  0.3017],
         [ 0.1898,  0.3208, -0.2315,  ...,  0.3714,  0.2478,  0.8048],
         [ 0.2071,  0.5036, -0.0485,  ...,  1.2175, -0.2292,  0.8582],
         [-3.4278,  0.0645, -0.1427,  ...,  0.0658, -0.4367,  0.3834]]],
       grad_fn=<NativeLayerNormBackward0>)
Length:      1
File:        /opt/anaconda3/envs/thellmbook/lib/python3.11/site-packages/torch/__init__.py
Docstring:   <no docstring>

In [28]:
output.shape

torch.Size([1, 4, 384])

# <ONLY For TEST 

In [29]:
def parameter_inspector(**kwargs):
    for key, value in kwargs.items():
        print(f"{key}: {value}")

In [30]:
parameter_inspector(**tokens)

input_ids: tensor([[    1, 31414,   232,     2]])
token_type_ids: tensor([[0, 0, 0, 0]])
attention_mask: tensor([[1, 1, 1, 1]])


In [31]:
tokens

{'input_ids': tensor([[    1, 31414,   232,     2]]), 'token_type_ids': tensor([[0, 0, 0, 0]]), 'attention_mask': tensor([[1, 1, 1, 1]])}

In [32]:
w=tokens['input_ids'][0]
print(w)


tensor([    1, 31414,   232,     2])


In [33]:
import numpy as np

In [34]:
three_shape=np.random.randint(0, 255, size=(1,2,2))
three_shape

array([[[80, 98],
        [25, 37]]])

In [35]:
three_shape[0][1][1]

37

# ONLY For TEST >

In [36]:
type(output)

torch.Tensor

In [37]:
print(output)

tensor([[[-3.4816,  0.0861, -0.1819,  ..., -0.0612, -0.3911,  0.3017],
         [ 0.1898,  0.3208, -0.2315,  ...,  0.3714,  0.2478,  0.8048],
         [ 0.2071,  0.5036, -0.0485,  ...,  1.2175, -0.2292,  0.8582],
         [-3.4278,  0.0645, -0.1427,  ...,  0.0658, -0.4367,  0.3834]]],
       grad_fn=<NativeLayerNormBackward0>)


In [38]:
output.shape

torch.Size([1, 4, 384])

In [39]:
for token in tokens['input_ids'][0]:
    print(f'id is:{token} \t  equivalent:{tokenizer.decode(token)}')

id is:1 	  equivalent:[CLS]
id is:31414 	  equivalent:Hello
id is:232 	  equivalent: world
id is:2 	  equivalent:[SEP]


In [40]:
output

tensor([[[-3.4816,  0.0861, -0.1819,  ..., -0.0612, -0.3911,  0.3017],
         [ 0.1898,  0.3208, -0.2315,  ...,  0.3714,  0.2478,  0.8048],
         [ 0.2071,  0.5036, -0.0485,  ...,  1.2175, -0.2292,  0.8582],
         [-3.4278,  0.0645, -0.1427,  ...,  0.0658, -0.4367,  0.3834]]],
       grad_fn=<NativeLayerNormBackward0>)

# Text Embeddings (For Sentences and Whole Documents)

# This is the alternative and shorter method for above method

In [41]:
from sentence_transformers import SentenceTransformer

# Load model
model = SentenceTransformer('sentence-transformers/all-mpnet-base-v2')

# Convert text to text embeddings
vector = model.encode("Best movie ever!")

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
/opt/anaconda3/envs/thellmbook/lib/python3.11/site-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


In [42]:
print(vector)
print(type(vector))

[-2.02204157e-02  4.57697064e-02 -1.26636969e-02 -3.37990746e-03
 -2.54909531e-03 -3.31097841e-03 -4.58366945e-02  2.32390966e-02
 -3.12585458e-02 -3.15589085e-02 -2.19783448e-02  3.67820193e-03
  3.39106540e-03 -1.88130792e-02 -2.65821610e-02 -1.45892687e-02
  3.20066959e-02 -3.66491778e-03  1.75410379e-02  5.43209650e-02
 -3.96662094e-02  9.93637461e-03 -3.27205025e-02 -1.62910372e-02
  6.36529410e-03 -1.24459155e-02  1.04737058e-02  3.08674518e-02
  7.47453189e-03 -3.86319980e-02 -6.19924366e-02 -3.25308964e-02
 -1.40269026e-02  5.16586974e-02  1.69651707e-06 -1.74110086e-04
 -1.27880741e-03 -4.15053181e-02  5.00346022e-03  4.58600633e-02
 -2.85337642e-02  5.02370670e-02 -1.75183769e-02 -3.98821750e-04
  1.63311474e-02  5.89936338e-02 -2.21729912e-02 -4.00975049e-02
 -6.60440046e-03 -2.97632832e-02 -3.18221673e-02 -1.77945606e-02
 -1.54362870e-02 -3.44515033e-02  6.25759810e-02  2.34394567e-03
  3.44158337e-02 -8.25460255e-03  7.44929686e-02 -3.81285213e-02
 -6.80808797e-02 -4.23290

In [43]:
vector.shape

(768,)

# Word Embeddings Beyond LLMs


In [44]:
import gensim.downloader as api

# Download embeddings (66MB, glove, trained on wikipedia, vector size: 50)
# Other options include "word2vec-google-news-300"
# More options at https://github.com/RaRe-Technologies/gensim-data
model = api.load("glove-wiki-gigaword-50")

In [45]:
api.info()

{'corpora': {'semeval-2016-2017-task3-subtaskBC': {'num_records': -1,
   'record_format': 'dict',
   'file_size': 6344358,
   'reader_code': 'https://github.com/RaRe-Technologies/gensim-data/releases/download/semeval-2016-2017-task3-subtaskB-eng/__init__.py',
   'license': 'All files released for the task are free for general research use',
   'fields': {'2016-train': ['...'],
    '2016-dev': ['...'],
    '2017-test': ['...'],
    '2016-test': ['...']},
   'description': 'SemEval 2016 / 2017 Task 3 Subtask B and C datasets contain train+development (317 original questions, 3,169 related questions, and 31,690 comments), and test datasets in English. The description of the tasks and the collected data is given in sections 3 and 4.1 of the task paper http://alt.qcri.org/semeval2016/task3/data/uploads/semeval2016-task3-report.pdf linked in section “Papers” of https://github.com/RaRe-Technologies/gensim-data/issues/18.',
   'checksum': '701ea67acd82e75f95e1d8e62fb0ad29',
   'file_name': 'se

In [46]:
model.most_similar([model['king']], topn=11)

[('king', 1.0000001192092896),
 ('prince', 0.8236179351806641),
 ('queen', 0.7839044332504272),
 ('ii', 0.7746230363845825),
 ('emperor', 0.7736247777938843),
 ('son', 0.766719400882721),
 ('uncle', 0.7627150416374207),
 ('kingdom', 0.7542160749435425),
 ('throne', 0.7539914846420288),
 ('brother', 0.7492412328720093),
 ('ruler', 0.7434254288673401)]

In [47]:
model.most_similar??

Signature:
model.most_similar(
    positive=None,
    negative=None,
    topn=10,
    clip_start=0,
    clip_end=None,
    restrict_vocab=None,
    indexer=None,
)
Source:   
    def most_similar(
            self, positive=None, negative=None, topn=10, clip_start=0, clip_end=None,
            restrict_vocab=None, indexer=None,
        ):
        """Find the top-N most similar keys.
        Positive keys contribute positively towards the similarity, negative keys negatively.

        This method computes cosine similarity between a simple mean of the projection
        weight vectors of the given keys and the vectors for each key in the model.
        The method corresponds to the `word-analogy` and `distance` scripts in the original
        word2vec implementation.

        Parameters
        ----------
        positive : list of (str or int or ndarray) or list of ((str,float) or (int,float) or (ndarray,float)), optional
            List of keys that contribute positively. If tuple, s

In [48]:
model.most_similar([model['bridge']],topn=60)

[('bridge', 0.9999998807907104),
 ('road', 0.8527507185935974),
 ('bridges', 0.8246731758117676),
 ('tunnel', 0.8163517713546753),
 ('connecting', 0.7905849814414978),
 ('crosses', 0.7779373526573181),
 ('along', 0.7731157541275024),
 ('railway', 0.7710516452789307),
 ('crossing', 0.7682911157608032),
 ('rail', 0.761237382888794),
 ('highway', 0.7581579089164734),
 ('gate', 0.7550985217094421),
 ('canal', 0.7430083155632019),
 ('route', 0.7392379641532898),
 ('lane', 0.735925555229187),
 ('viaduct', 0.7353712916374207),
 ('constructed', 0.7335984110832214),
 ('embankment', 0.7333869338035583),
 ('built', 0.7328950762748718),
 ('roadway', 0.731002926826477),
 ('river', 0.7267074584960938),
 ('east', 0.7251012921333313),
 ('roads', 0.7232987880706787),
 ('junction', 0.7231907844543457),
 ('railroad', 0.722453773021698),
 ('parkway', 0.7220357656478882),
 ('adjacent', 0.7217997908592224),
 ('entrance', 0.7199317216873169),
 ('line', 0.7192555069923401),
 ('connects', 0.7173622250556946),


# Recommending songs by embeddings

In [88]:
import pandas as pd
from urllib import request


In [89]:
# Get the playlist dataset file
data = request.urlopen('https://storage.googleapis.com/maps-premium/dataset/yes_complete/train.txt')

# Parse the playlist dataset file. Skip the first two lines as
# they only contain metadata
lines = data.read().decode("utf-8").split('\n')[2:]


In [90]:
# Remove playlists with only one song
playlists = [s.rstrip().split() for s in lines if len(s.split()) > 1]

In [91]:
# Load song metadata
songs_file = request.urlopen('https://storage.googleapis.com/maps-premium/dataset/yes_complete/song_hash.txt')
songs_file = songs_file.read().decode("utf-8").split('\n')
songs = [s.rstrip().split('\t') for s in songs_file]
songs_df = pd.DataFrame(data=songs, columns = ['id', 'title', 'artist'])
songs_df = songs_df.set_index('id')

In [103]:
songs_df.head(10)

,title,artist
id,,
0,Gucci Time (w\/ Swizz Beatz),Gucci Mane
1,Aston Martin Music (w\/ Drake & Chrisette Mich...,Rick Ross
2,Get Back Up (w\/ Chris Brown),T.I.
3,Hot Toddy (w\/ Jay-Z & Ester Dean),Usher
4,Whip My Hair,Willow
5,Down On Me (w\/ 50 Cent),Jeremih
6,Black And Yellow,Wiz Khalifa
7,Blowing Me Kisses,Soulja Boy
8,Lay It Down,Lloyd


In [92]:
print( 'Playlist #1:\n ', playlists[0], '\n')
print( 'Playlist #2:\n ', playlists[1])

Playlist #1:
  ['0', '1', '2', '3', '4', '5', '6', '7', '8', '9', '10', '11', '12', '13', '14', '15', '16', '17', '18', '19', '20', '21', '22', '23', '24', '25', '26', '27', '28', '29', '30', '31', '32', '33', '34', '35', '36', '37', '38', '39', '40', '41', '2', '42', '43', '44', '45', '46', '47', '48', '20', '49', '8', '50', '51', '52', '53', '54', '55', '56', '57', '25', '58', '59', '60', '61', '62', '3', '63', '64', '65', '66', '46', '47', '67', '2', '48', '68', '69', '70', '57', '50', '71', '72', '53', '73', '25', '74', '59', '20', '46', '75', '76', '77', '59', '20', '43'] 

Playlist #2:
  ['78', '79', '80', '3', '62', '81', '14', '82', '48', '83', '84', '17', '85', '86', '87', '88', '74', '89', '90', '91', '4', '73', '62', '92', '17', '53', '59', '93', '94', '51', '50', '27', '95', '48', '96', '97', '98', '99', '100', '57', '101', '102', '25', '103', '3', '104', '105', '106', '107', '47', '108', '109', '110', '111', '112', '113', '25', '63', '62', '114', '115', '84', '116', '117',

In [93]:
from gensim.models import Word2Vec

# Train our Word2Vec model
model = Word2Vec(
    playlists, vector_size=32, window=20, negative=50, min_count=1, workers=4
)

In [94]:
song_id = 2172

# Ask the model for songs similar to song #2172
model.wv.most_similar(positive=str(song_id))

[('3167', 0.999354362487793),
 ('10084', 0.9972447156906128),
 ('2976', 0.9966879487037659),
 ('5586', 0.9966681003570557),
 ('2704', 0.9966265559196472),
 ('3094', 0.9965333938598633),
 ('2640', 0.9962555766105652),
 ('2849', 0.9960620403289795),
 ('6685', 0.996025025844574),
 ('1922', 0.9957937598228455)]

In [95]:
print(songs_df.iloc[2172])

title     Fade To Black
artist        Metallica
Name: 2172 , dtype: object


In [96]:
import numpy as np

def print_recommendations(song_id):
    similar_songs = np.array(
        model.wv.most_similar(positive=str(song_id),topn=5)
    )[:,0]
    return  songs_df.iloc[similar_songs]

# Extract recommendations
print_recommendations(2172)

,title,artist
id,,
3167,Unchained,Van Halen
10084,Detroit Rock City,Kiss
2976,I Don't Know,Ozzy Osbourne
5586,The Last In Line,Dio
2704,Over The Mountain,Ozzy Osbourne


In [97]:
print_recommendations(2172)

,title,artist
id,,
3167,Unchained,Van Halen
10084,Detroit Rock City,Kiss
2976,I Don't Know,Ozzy Osbourne
5586,The Last In Line,Dio
2704,Over The Mountain,Ozzy Osbourne


In [98]:
print_recommendations(842)

,title,artist
id,,
413,If I Ruled The World (Imagine That) (w\/ Laury...,Nas
12335,King Of The Dancehall,Beenie Man
330,Hate It Or Love It (w\/ 50 Cent),The Game
27081,"Give Me Everything (w\/ Ne-Yo, Afrojack & Nayer)",Pitbull
5668,How We Do (w\/ 50 Cent),The Game
